# 📊 EDA PLAYBOOK — TOOLKIT PROFESIONAL

> **Guía completa de Análisis Exploratorio de Datos**  
> Cubre desde el contexto del dataset hasta la generación de insights accionables.

---

| Sección | Descripción |
|---|---|
| 1 | Dataset Context |
| 2 | Necessary Libraries |
| 3 | Dataset Loading |
| 4 | Data Overview |
| 5 | Data Quality Assessment |
| 6 | Missing Values Analysis |
| 7 | Duplicate Detection |
| 8 | Univariate Analysis |
| 9 | Categorical Analysis |
| 10 | Bivariate Analysis |
| 11 | Correlation Analysis |
| 12 | Outlier Detection |
| 13 | Distribution Analysis |
| 14 | Feature Relationships |
| 15 | Data Leakage Check |
| 16 | Hypothesis Generation |
| 17 | Insights & Conclusions |

---
## 1️⃣ Dataset Context

Antes de tocar el código, documenta el contexto del dataset. Esta sección es **la más importante** del EDA: sin contexto, el análisis técnico no tiene sentido.

**Qué buscar:**
- ¿Qué representa cada fila?
- ¿Hay agregaciones o es dato crudo?
- ¿La granularidad es la esperada?

| Interpretación | Descripción |
|---|---|
| **Técnica** | Comprender la estructura semántica del dataset |
| **Negocio** | Explicar qué representa cada registro en términos del dominio |

> **Ejemplo:** *"Cada fila representa una compra realizada por un usuario en la plataforma e-commerce."*

In [ ]:
"""
DATASET CONTEXT
===============
Source            : 
Date extracted    : 
Business domain   : 
Unit of observation: 
Goal of analysis  : 
"""

---
## 2️⃣ Necessary Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Encoding
from sklearn.preprocessing import LabelEncoder

# Configuration
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

print("✅ Libraries loaded")

---
## 3️⃣ Dataset Loading

**Qué buscar:**
- Tamaño del dataset (filas × columnas)
- Columnas inesperadas o con nombres extraños
- Problemas de encoding (caracteres especiales)

In [ ]:
# Ajusta la ruta a tu archivo
df = pd.read_csv("dataset.csv")

print(f"Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

---
## 4️⃣ Data Overview

**Qué buscar:**

| Check | Qué detectar |
|---|---|
| `dtypes` | Tipos incorrectos (ej: fecha como object, número como string) |
| `missing` | Valores faltantes visibles |
| `rangos` | Valores imposibles (edad = -5, precio = 0) |
| `cardinalidad` | Categorías con demasiados valores únicos |

In [ ]:
print("--- HEAD ---")
display(df.head())

In [ ]:
print("--- TAIL ---")
display(df.tail())

In [ ]:
print("--- INFO ---")
df.info()

In [ ]:
print("--- DESCRIBE (Numéricas) ---")
display(df.describe())

In [ ]:
print("--- DESCRIBE (Categóricas) ---")
display(df.describe(include="object"))

---
## 5️⃣ Data Quality Assessment

**Qué buscar:**
- Columnas con **>40% missing** → considerar drop o imputación especial
- Variables con **cardinalidad extrema** → posibles IDs ocultos o columnas de texto libre
- Columnas que son **IDs disfrazados** de features

In [ ]:
def data_quality_report(df):
    report = pd.DataFrame({
        "dtype"       : df.dtypes,
        "missing"     : df.isna().sum(),
        "missing_pct" : (df.isna().sum() / len(df) * 100).round(2),
        "unique"      : df.nunique(),
        "unique_pct"  : (df.nunique() / len(df) * 100).round(2)
    })
    report = report.sort_values("missing_pct", ascending=False)
    return report

display(data_quality_report(df))

---
## 6️⃣ Missing Values Analysis

**Interpretación técnica:** Identificar patrones de missingness (MCAR / MAR / MNAR).  
**Interpretación negocio:** Traducir los nulos a contexto real.

> **Ejemplo:** *"El 35% de los clientes no tienen edad registrada. Puede indicar fallo en formulario o campo opcional."*

In [ ]:
missing     = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "missing"    : missing,
    "missing_pct": missing_pct
})

# Solo mostrar columnas con al menos 1 nulo
display(missing_df[missing_df["missing"] > 0])

In [ ]:
# Visualización de missing values
cols_with_missing = missing_df[missing_df["missing"] > 0].index.tolist()

if cols_with_missing:
    plt.figure(figsize=(10, 4))
    missing_df[missing_df["missing"] > 0]["missing_pct"].sort_values().plot(
        kind="barh", color="salmon", edgecolor="black"
    )
    plt.title("% Missing por columna", fontsize=14)
    plt.xlabel("% Missing")
    plt.axvline(40, color="red", linestyle="--", label="Umbral 40%")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("✅ No hay valores missing en el dataset.")

---
## 7️⃣ Duplicate Detection

**Qué buscar:**
- Filas completamente duplicadas
- Duplicados parciales (mismo ID con datos distintos)

In [ ]:
n_duplicates = df.duplicated().sum()
pct = (n_duplicates / len(df) * 100).round(2)

print(f"Filas duplicadas: {n_duplicates:,} ({pct}% del total)")

if n_duplicates > 0:
    print("\nEjemplo de duplicados:")
    display(df[df.duplicated(keep=False)].head(10))

---
## 8️⃣ Univariate Analysis — Variables Numéricas

**Qué buscar:**
- **Skewness**: distribuciones muy asimétricas pueden requerir transformación
- **Outliers**: valores extremos que distorsionan el análisis
- **Rangos**: valores fuera del rango lógico del negocio

In [ ]:
# Histogramas de todas las variables numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    df[numeric_cols].hist(figsize=(14, 10), bins=30, edgecolor="black", color="steelblue")
    plt.suptitle("Distribución de Variables Numéricas", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No hay columnas numéricas.")

In [ ]:
# KDE + Histograma por columna
# Reemplaza 'column' por el nombre de tu variable de interés
col = numeric_cols[0] if numeric_cols else None

if col:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(df[col], kde=True, ax=axes[0], color="steelblue")
    axes[0].set_title(f"Histograma + KDE: {col}")

    sns.boxplot(x=df[col], ax=axes[1], color="lightcoral")
    axes[1].set_title(f"Boxplot: {col}")

    plt.tight_layout()
    plt.show()

    print(f"\nInterpretación técnica: skew = {df[col].skew():.2f}")
    print("Interpretación negocio: [Escribe tu conclusión aquí]")

---
## 9️⃣ Categorical Analysis

**Qué buscar:**
- **Clases dominantes**: una categoría que concentra >80% → puede indicar desbalance
- **Clases raras**: categorías con <1% de ocurrencia → candidatas a agrupar en "Otros"

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_cols:
    print(f"\n📌 {col} ({df[col].nunique()} valores únicos)")
    display(df[col].value_counts(normalize=True).mul(100).round(2).rename("%").to_frame())

In [ ]:
# Countplot para cada variable categórica
for col in cat_cols:
    if df[col].nunique() <= 20:  # Solo si tiene pocos valores únicos
        plt.figure(figsize=(8, 4))
        order = df[col].value_counts().index
        sns.countplot(x=col, data=df, order=order, palette="Blues_r", edgecolor="black")
        plt.title(f"Distribución: {col}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️ {col} tiene {df[col].nunique()} valores únicos — considera agrupar o revisar.")

---
## 🔟 Bivariate Analysis

**Tipos de análisis bivariado:**

| Combinación | Gráfico | Buscar |
|---|---|---|
| Num vs Num | Scatter | Correlaciones, clusters |
| Cat vs Num | Boxplot | Diferencias entre grupos |
| Cat vs Cat | Crosstab / Heatmap | Asociaciones, patrones |

In [ ]:
# Numeric vs Numeric
# Reemplaza col1 y col2 por tus variables
if len(numeric_cols) >= 2:
    col1, col2 = numeric_cols[0], numeric_cols[1]

    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=col1, y=col2, data=df, alpha=0.6)
    plt.title(f"Scatter: {col1} vs {col2}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Category vs Numeric
if cat_cols and numeric_cols:
    cat_col = cat_cols[0]
    num_col = numeric_cols[0]

    if df[cat_col].nunique() <= 15:
        plt.figure(figsize=(10, 5))
        order = df.groupby(cat_col)[num_col].median().sort_values(ascending=False).index
        sns.boxplot(x=cat_col, y=num_col, data=df, order=order, palette="Set2")
        plt.title(f"{num_col} por {cat_col}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
# Category vs Category
if len(cat_cols) >= 2:
    col1, col2 = cat_cols[0], cat_cols[1]

    if df[col1].nunique() <= 15 and df[col2].nunique() <= 15:
        ct = pd.crosstab(df[col1], df[col2], normalize="index").round(2)
        print(f"Crosstab (normalizado por fila): {col1} vs {col2}")
        display(ct)

        plt.figure(figsize=(8, 5))
        sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues")
        plt.title(f"{col1} vs {col2}")
        plt.tight_layout()
        plt.show()

---
## 1️⃣1️⃣ Correlation Analysis

**Qué buscar:**
- Correlaciones **> 0.7** o **< -0.7**: relación fuerte entre variables
- Variables **redundantes**: dos features que miden lo mismo → eliminar una
- Correlación con el **target**: identifica las features más predictivas

In [ ]:
if len(numeric_cols) > 1:
    corr = df.corr(numeric_only=True)

    plt.figure(figsize=(12, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool))  # Solo triángulo inferior
    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        mask=mask,
        linewidths=0.5
    )
    plt.title("Correlation Matrix", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Pares altamente correlacionados
    print("\n🔴 Pares con correlación absoluta > 0.7:")
    high_corr = (
        corr.abs()
        .where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
        .stack()
        .sort_values(ascending=False)
    )
    display(high_corr[high_corr > 0.7].rename("correlation").to_frame())
else:
    print("No hay suficientes columnas numéricas para la matriz de correlación.")

---
## 1️⃣2️⃣ Outlier Detection

**Método IQR** (robusto, no asume normalidad):
- `lower = Q1 - 1.5 × IQR`
- `upper = Q3 + 1.5 × IQR`

**Qué buscar:**
- Outliers legítimos (eventos extremos reales) vs errores de datos
- Porcentaje de outliers por variable → si >5%, investigar

In [ ]:
def detect_outliers_iqr(df, col):
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[col] < lower) | (df[col] > upper)]

# Resumen de outliers por columna numérica
outlier_summary = []
for col in numeric_cols:
    n_out = len(detect_outliers_iqr(df, col))
    pct   = (n_out / len(df) * 100).round(2)
    outlier_summary.append({"column": col, "n_outliers": n_out, "pct_outliers": pct})

outlier_df = pd.DataFrame(outlier_summary).sort_values("pct_outliers", ascending=False)
display(outlier_df)

In [ ]:
# Boxplots para visualizar outliers en todas las variables numéricas
if numeric_cols:
    n     = len(numeric_cols)
    ncols = 3
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3))
    axes = axes.flatten()

    for i, col in enumerate(numeric_cols):
        sns.boxplot(x=df[col], ax=axes[i], color="lightblue")
        axes[i].set_title(col)

    # Ocultar ejes vacíos
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Outliers por Variable (IQR)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 1️⃣3️⃣ Distribution Analysis

**Interpretación del skewness:**

| Skew | Interpretación |
|---|---|
| ~0 | Distribución normal |
| > 1 | Muy sesgada a la derecha (cola larga en valores altos) |
| < -1 | Muy sesgada a la izquierda (cola larga en valores bajos) |

In [ ]:
skew_df = df[numeric_cols].skew().sort_values(key=abs, ascending=False).round(3)

print("📐 Skewness por variable:")
display(skew_df.rename("skew").to_frame())

high_skew = skew_df[skew_df.abs() > 1]
if not high_skew.empty:
    print("\n⚠️ Variables con skew > 1 (considerar transformación log/sqrt):")
    display(high_skew.rename("skew").to_frame())

---
## 1️⃣4️⃣ Feature Relationships

**Qué buscar:**
- **Features redundantes**: dos variables que miden prácticamente lo mismo → eliminar una para evitar multicolinealidad
- **Variables proxy**: una feature que actúa como sustituto de otra (ej: código postal ≈ nivel socioeconómico)
- **Interacciones**: combinaciones de features que juntas explican mejor el target

In [ ]:
# Pairplot para visualizar relaciones entre las primeras N variables numéricas
N = 5  # Ajusta según cuántas variables quieras comparar
cols_to_plot = numeric_cols[:N]

if len(cols_to_plot) >= 2:
    sns.pairplot(df[cols_to_plot], diag_kind="kde", plot_kws={"alpha": 0.4})
    plt.suptitle("Pairplot — Feature Relationships", y=1.02, fontsize=14)
    plt.show()
else:
    print("No hay suficientes variables numéricas para el pairplot.")

In [ ]:
# Notas de feature relationships
feature_notes = """
Features redundantes identificadas:
- [Feature A] y [Feature B] tienen correlación de X — considerar eliminar una.

Variables proxy detectadas:
- [Ej: código postal puede ser proxy de nivel socioeconómico]

Interacciones potenciales:
- [Ej: edad × categoría de producto podría ser predictiva]
"""
print(feature_notes)

---
## 1️⃣5️⃣ Data Leakage Check

El **data leakage** es uno de los errores más peligrosos en ML: el modelo aprende información que no tendría disponible en producción, generando métricas artificialmente buenas.

**Preguntas clave:**
1. ¿Hay variables que contienen **información del futuro**?
2. ¿Hay variables **derivadas del target**?
3. ¿Hay variables que **solo existen tras el evento** que queremos predecir?

In [ ]:
# Checklist de Data Leakage
leakage_checklist = {
    "Variables con información del futuro"    : "[ ] Revisar columnas de fecha, estados post-evento",
    "Variables derivadas del target"          : "[ ] Buscar columnas calculadas a partir del target",
    "Variables que solo existen tras el evento": "[ ] Ej: fecha de cancelación en predicción de churn",
    "Features con correlación perfecta (1.0)" : "[ ] Revisar la correlation matrix anterior",
    "Columnas de ID o timestamp filtrados"    : "[ ] Verificar que no incluyen señal del target"
}

print("🔍 DATA LEAKAGE CHECKLIST")
print("=" * 50)
for check, note in leakage_checklist.items():
    print(f"\n{'•'} {check}")
    print(f"  {note}")

---
## 1️⃣6️⃣ Hypothesis Generation

Basándote en lo observado en el EDA, formula hipótesis de negocio que puedan guiar el modelado o análisis más profundo.

> **Ejemplo de hipótesis:**
> - *"Usuarios jóvenes (18-25) tienen mayor tasa de conversión en mobile."*
> - *"Las promociones de fin de semana generan picos de compra no sostenibles."*

In [ ]:
hypotheses = [
    # Añade tus hipótesis aquí
    "H1: [Variable X] tiene relación positiva con [Variable Y]",
    "H2: [Segmento A] se comporta diferente a [Segmento B] respecto a [métrica]",
    "H3: [Evento/Condición] aumenta/reduce significativamente [target]",
]

print("💡 HIPÓTESIS GENERADAS")
print("=" * 50)
for h in hypotheses:
    print(f"  • {h}")

---
## 1️⃣7️⃣ Insights & Conclusions

El objetivo final del EDA es **transformar observaciones técnicas en conclusiones accionables**.

> **Ejemplo:** *"El 70% de las ventas provienen de usuarios recurrentes → priorizar retención sobre adquisición."*

In [ ]:
insights = {
    "Calidad de datos": [
        "[Ej: 35% de nulos en columna 'edad' — requiere imputación o análisis por separado]",
        "[Ej: X% de registros duplicados eliminados]"
    ],
    "Distribuciones": [
        "[Ej: La variable 'ingresos' tiene fuerte skew derecho — considerar log-transform]",
        "[Ej: Outliers en 'precio' pueden representar transacciones B2B]"
    ],
    "Relaciones entre variables": [
        "[Ej: Alta correlación entre feature A y B — candidatas a reducción de dimensionalidad]",
        "[Ej: La categoría X explica el mayor % de varianza en el target]"
    ],
    "Insights de negocio": [
        "[Ej: El 70% de las ventas provienen de usuarios recurrentes]",
        "[Ej: El segmento premium representa el 20% de usuarios pero el 60% de ingresos]"
    ],
    "Próximos pasos recomendados": [
        "[Ej: Feature engineering sobre variables de fecha]",
        "[Ej: Imputación de missings con modelo KNN en columna X]",
        "[Ej: Investigar outliers en columna Y con el equipo de negocio]"
    ]
}

print("📋 RESUMEN DE INSIGHTS")
print("=" * 55)
for section, items in insights.items():
    print(f"\n🔹 {section.upper()}")
    for item in items:
        print(f"   • {item}")

---

## ✅ EDA Completado

| Sección | Estado |
|---|---|
| Dataset Context | ⬜ |
| Libraries | ⬜ |
| Loading | ⬜ |
| Data Overview | ⬜ |
| Quality Assessment | ⬜ |
| Missing Values | ⬜ |
| Duplicates | ⬜ |
| Univariate Analysis | ⬜ |
| Categorical Analysis | ⬜ |
| Bivariate Analysis | ⬜ |
| Correlation | ⬜ |
| Outliers | ⬜ |
| Distribution | ⬜ |
| Feature Relationships | ⬜ |
| Data Leakage | ⬜ |
| Hypotheses | ⬜ |
| Insights | ⬜ |

> Cambia ⬜ por ✅ a medida que completes cada sección.